# Object Detection using Faster RCNN

# Step 1 -Import Libraries

In [2]:
import tensorflow as tf
from tensorflow import keras
import numpy as np
import cv2
import matplotlib.pyplot as plt
import random

2024-11-03 17:11:31.731928: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2024-11-03 17:11:31.744417: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2024-11-03 17:11:31.776725: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1730635891.845068    4223 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1730635891.856889    4223 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-11-03 17:11:31.899650: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU ins

ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

# Step 2 -Construct the backbone

In [3]:
def build_backbone(input_shape = (224 , 224 , 3)):
    inputs = keras.Input(shape = input_shape)
    x = keras.layers.Conv2D(64 , (3,3) , activation = 'relu' , padding = 'same')(inputs)
    x = keras.layers.Conv2D(128, (2,2))(x)
    x = keras.layers.MaxPooling2D((2,2))(x)
    x = keras.layers.Conv2D(256 , (3,3) , activation = 'relu' , padding = 'same')(x)
    x = keras.layers.MaxPooling2D((2,2))(x)
    outputs = keras.layers.Conv2D(512 , (3,3) , activation = 'relu' , padding = 'same')(x)
    return keras.Model(inputs , outputs)

backbone = build_backbone()

2024-10-29 19:13:53.091662: I tensorflow/core/common_runtime/process_util.cc:146] Creating new thread pool with default inter op setting: 2. Tune using inter_op_parallelism_threads for best performance.


# Step 3 -construct the Region Proposal Network(RPN)

In [ ]:
def build_rpm(feature_map):
    rpn_conv = keras.layers.Conv2D(512 , (3,3) , paddings = "same" , activation = 'relu')(feature_map)
    rpn_class = keras.layers.Conv2D(9*2 , (1,1) , activation = "sigmoid")(rpn_conv) # 9 anchor, 2 classes (object / non-object)
    rpn_bbox = keras.layers.Conv2D(9*4 , (1,1))(rpn_conv) # 9 anchors , 4 bounding box values
    return rpn_class , rpn_bbox

# Step 4 -Implement the ROI Pooling layer

The ROI Pooling Layer extracts fixed size feature maps from ROIs

In [ ]:
class ROIPoolingLayer(tf.keras.layers.Layer):
    def __init__(self , pool_size , **kwargs):
        super(ROIPoolingLayer , self).__init__(**kwargs)
        self.pool_size = pool_size
        
    def call(self , inputs):
        feature_map, rois = inputs
        rois = tf.reshape(rois , (-1 ,4))
        batch_indices = tf.zeros((tf.shape(rois)[0] , ), dtype = tf.int32)
        pooled_rois = tf.image.crop_and_resize(
            feature_map , rois , batch_indices = batch_indices, crop_size = self.pool_size)
        
        return pooled_rois
    def compute_output_shape(self , input_shape):
        feature_map_shape , rois_shape = input_shape
        num_rois = rois_shape[0]
        return (num_rois , self.pool_size[0] , feature_map_shape[-1])
    
roi_pooling_layer = ROIPoolingLayer((7,7))        

# Step 5: Create classificaton and regression Heads

After pooling features, we create dense layers to classify objects and predict bounding box adjustments.

In [ ]:
def build_heads(pooled_rois , num_classes):
    flatten = keras.layers.Flatten()(pooled_rois)
    dense = keras.layers.Dense(1024 , activation = "relu")(flatten)
    classifier = keras.layers.Dense(num_classes , activation = "softmax")(dense)
    bbox_regressor = keras.layers.Dense(num_classes * 4)(dense) # 4 perbounding box
    return classifier , bbox_regressor

# Step 6: Assembling the Complete Faster R-CNN Model:

we build all the components together to build the final Faster R-CNN model

In [ ]:
# Assemble the complete Faster R-CNN Model
def build_faster_rcnn(num_classes , input_shape = (224 , 224 , 3)):
    input_image = keras.Input(shape = input_shape)
    rois = keras.Input(shape = (None , 4)) # ROI input
    
    # Backbone(feature extraction)
    backbone = build_backbone(input_shape)
    feature_map = backbone(input_image)
    
    # Region Proposal Network(RPN)
    rpn_class, rpn_bblox = build_rpn(feature_map)
    
    # ROI Pooling
    pooled_rois = roi_pooling_layer([feature_map , rois])
    
    # classification and regression head
    classifier , bbox_regressor = build_heads(pooled_rois , num_classes)
    
    # build the complete model
    model = keras.Model(inputs = [input_image , rois], outputs = [rpn_class , rpn_bbox, classifier , bbox_regressor])

# Step 7 -Printing Summary of the model:

The summary of the model can be displayed

In [ ]:
# Instantiate the model
num_classes = 80 # Example number of classes for COCO Dataset
faster_rcnn = build_faster_rcnn(num_classes)
faster_rcnn.summary()

# Step 8 -Calculating IOU

In [ ]:
def calculate_iou(box1 , box2):
    ymin1,xmin1 , ymax1 , xmax1 = box1
    ymin2, xmin2 , ymax2, xmax2 = box2
    inter_xmin = max(xmin1 , xmin2)
    inter_ymin = max(ymin1 , ymin2)
    inter_xmax = min(xmax1 , xmax2)
    inter_ymax = min(ymax1 , ymax2)
    inter_area = max(0 , inter_xmax - inter_xmin) * max(o, inter_ymax - inter_ymin)
    area1 = (xmax1 - xmin1) * (ymax1 - ymin1)
    area2 = (xmax2 - xmin2) * (ymax2 - ymin2)
    union_area = area1 + area2 - inter_area
    iou = inter_area / union_area if union_area != 0 else 0
    return iou
    

# Step 9 -Calculating Precision and Recall

In [ ]:
# calculate precision and recall
def calculate_precision_recall(pred_boxes , pred_scores, pred_labels, gt_boxes, gt_labels, iou_threshold = 0.5):
    tp, fp = 0, 0
    matched_gt = set()
    for i, pred_box in enumerate(pred_boxes):
        if pred_scores[i] < iou_threshold:
            continue
        best_iou = 0
        best_gt_index = -1
        for j, gt_box in enumerate(gt_boxes):
            if pred_labels[i] == gt_labels[j] and j not in matched_gt:
                iou = calculate_iou(pred_box , gt_box)
                if iou> best_iou:
                    best_iou = iou
                    best_gt_index = j
        if best_iou >= iou_threshold:
            matched_gt.add(best_gt_index)
            tp += 1
        else:
            fp +=1
            
    fn=len(gt_boxes) - len(matched_gt)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    
    return precision , recall
                        
        

# Step 10- Calculating Average Precision and mAP

In [1]:
# calculate the Average Precision
def calculate_ap(precisions , recalls):
    precisions = [0.0] + precisions + [0.0]
    recalls = [0.0] + recalls + [1.0]
    for i in range(len(precisions) - 1, 0 ,-1):
        precisions[i - 1] = max(precisions[i - 1], precisions[i])
    indices = sum((recalls[i] - recalls[i - 1]) * precisions[i] for i in indices)
    return calculate_ap

# Calculate the mean Average precision (mAP) access all classes
def calculate_map(predictions, ground_truths , num_classes, iou_threshold = 0.5):
    average_precisions = []
    for class_id in range(1 , num_classes): # Assuming class id=0 is background
        all_precisions = []
        for class_id in range(1, num_classes, iou_threshold = 0.5):
            all_precision = []
            all_recalls = []
            for pred, gt in zip(predictions, ground_truths):
                pred_boxes = [box for i, box in enumerate(pred['boxes']) if pred['labels'][i] == class_id]
                pred_scores = [score for i, score in enumerate(pred['scores']) if pred['labels'][i] == class_id]
                gt_boxes = [box for i, box in enumerate(gt['boxes']) if gt['labels'][i] == class_id]
                precision, recall = calculate_precision_recall(
                    pred_boxes, pred_scores, [class_id] * len(pred_boxes), gt_boxes, [class_id]*len(gt_boxes), iou_threshold
                )           
                all_precision.append(precision)
                all_recalls.append(recall)
            ap = calculate_ap(all_precisions , all_recalls)
            average_precisions.append(ap)
        mAP = np.mean(average_precisions)
        return mAP

# Step 11 -Sample Values for obtaining mAP

In [ ]:
# placeholder predictions and ground truths for testing
predictions = [
    {'boxes' : [[0.1, 0.1, 0.4, 0.4],[0.5,0.5,0.8,0.8]],'scores':[0.9, 0.7], 'labels':[1,2]}
]
ground_truths = [
    {'boxes': [[0.12 , 0.12, 0.42, 0.42],[0.5,0.5,0.8,0.8]],'labels':[1,2]}
]

# Number of classes in the dataset (e.g. 80 for COCO)
num_classes = 80

# Calculate the mean Average precision (mAP) access all classes
mean_ap = calculate_map(predictions , ground_truths, num_classes)
print("Mean Average Precision(mAP):{mean_ap:.2f} ")